# CKAN を検索して地図に表示する

この Showcase は、G空間情報センターの公開 CKAN で「河川」を検索し、選んだ GeoJSON を Rhinestone で解決してから、利用者所有の pyogrio / GeoPandas で表示します。

```text
configure -> search -> Result を選ぶ -> resolve -> Resource -> AccessPlan
    -> pyogrio -> GeoDataFrame.plot()
```

Rhinestone は検索・Resource の解決・Runtime への委譲を担います。データ読込後の表示や解析は downstream library の責務です。結果と公開状態は Provider 側で変わります。

## Setup

Colab ではこのセルを一度実行します。Rhinestone Core に GIS / Notebook 依存を追加するものではありません。

In [1]:
import importlib.util
import subprocess
import sys

packages = [
    "git+https://github.com/u-kitazawa/rhinestone.git@develop",
    "pyogrio==0.13.0",
    "geopandas==1.1.4",
    "folium==0.20.0",
    "ipywidgets",
]
if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

installation = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages],
    check=True,
)
print("Installed Showcase dependencies.")

Installed Showcase dependencies.


## 検索する

ここでは単一 Provider を構成します。複数 Provider を構成したとき、Rhinestone は Provider 横断の関連度順位を作りません。

In [2]:
from rhinestone import configure, sources

app = configure(sources=(sources.GEOSPATIAL_JP,))
results = app.search(text="河川", limit=20)

print(f"結果数: {len(results)}")
print("source groups:", results.keys())
print("diagnostics:", results.diagnostics)

結果数: 34
source groups: ('geospatial-jp',)
diagnostics: ()


## GeoJSON の候補を選ぶ

候補の形式は CKAN が広告した metadata をそのまま確認します。URL suffix や archive の中身から形式を推測しません。プルダウンを変更したら、次のセル以降を再実行してください。

In [3]:
from collections.abc import Mapping

import ipywidgets as widgets
from IPython.display import display


def advertised_geojson(result):
    raw_resource = result.raw_metadata.get("resource")
    if not isinstance(raw_resource, Mapping):
        return False
    advertised_format = raw_resource.get("format")
    return (
        isinstance(advertised_format, str) and advertised_format.casefold() == "geojson"
    )


candidates = [result for result in results if advertised_geojson(result)]
if not candidates:
    raise RuntimeError(
        "The CKAN search returned no explicitly advertised GeoJSON resources. "
        "Change the search term; this notebook will not infer a format or archive member."
    )

# This explicit public resource gives the committed GitHub preview a useful map.
# If the Provider no longer returns it, keep the Showcase interactive with the first candidate.
default_index = next(
    (
        index
        for index, result in enumerate(candidates)
        if result.provenance.resource_identifier
        == "78b3a6b9-f659-427a-bcc7-381074a38818"
    ),
    0,
)

for index, result in enumerate(candidates):
    print(
        f"[{index}] {result.title} | resource_id={result.provenance.resource_identifier} "
        f"| discovered_by={result.discovered_by} | target={result.target.source_id}"
    )

selector = widgets.Dropdown(
    options=[
        (f"{index}: {result.title} ({result.provenance.resource_identifier})", index)
        for index, result in enumerate(candidates)
    ],
    value=default_index,
    description="GeoJSON:",
    layout=widgets.Layout(width="95%"),
)
display(selector)

[0] 令和元年度　二級河川小坂川河川調査に伴う河川整備基本方針検討業務委託（その１） | resource_id=e8985aa9-1d5e-40d3-b691-7fc844d11d0e | discovered_by=geospatial-jp | target=geospatial-jp
[1] 島田川広域河川改修（推進費）第6工区 | resource_id=87de19ac-34c6-4801-a024-41de8f5613c5 | discovered_by=geospatial-jp | target=geospatial-jp
[2] 令和元年度　二級河川新中川　河川調査に伴う測量業務委託 | resource_id=3876bc2a-30e8-40b2-a809-987d72c0dd86 | discovered_by=geospatial-jp | target=geospatial-jp
[3] 平成29年度[第29-K2452-01]二級河川巴川（麻機遊水地）総合治水対策特定河川事業（防災・安全交付金）工事（加藤島エリア掘削築堤工） | resource_id=cc3b73e3-311b-4761-a378-809312939914 | discovered_by=geospatial-jp | target=geospatial-jp
[4] 令和元年度［30-D4651-01］（一）富士由比線県単橋梁改築に伴う設計業務委託（河川占用資料作成業務） | resource_id=9741f948-a4d5-4400-8502-6e9e5fe7e4d7 | discovered_by=geospatial-jp | target=geospatial-jp
[5] 令和元年度［第31-B0301-02号］ 二級河川波多打川ほか災害復旧工法委託（河川調査）に伴う測量・設計業務委託 | resource_id=91df48df-0813-449f-a07e-97ce361ac010 | discovered_by=geospatial-jp | target=geospatial-jp
[6] 島田川広域河川改修（補修）工事　第8工区 | resource_id=978624c1-5567-414c-be31-3d60002b8378 | disc

Dropdown(description='GeoJSON:', index=13, layout=Layout(width='95%'), options=(('0: 令和元年度\u3000二級河川小坂川河川調査に伴う…

## Resource と AccessPlan を確認する

`Result` を通常の解決パイプラインへ渡します。大きな raw response は表示せず、利用時に重要な metadata と provenance だけを確認します。

In [4]:
selected = candidates[selector.value]
resource = app.resolve(selected)

print("discovery source:", selected.discovered_by)
print("resolution target:", selected.target.source_id)
print("URI:", resource.uri)
print("format / media type:", resource.format, "/", resource.media_type)
print("AccessPlan:", resource.access_plan)
print(
    "metadata:",
    {
        "title": resource.metadata.title,
        "description": resource.metadata.description,
        "publisher": resource.metadata.publisher,
        "license": resource.metadata.license,
    },
)
print(
    "provenance:",
    {
        "provider": resource.provenance.provider,
        "dataset_identifier": resource.provenance.dataset_identifier,
        "resource_identifier": resource.provenance.resource_identifier,
        "adapter": resource.provenance.adapter,
        "original_url": resource.provenance.original_url,
    },
)

discovery source: geospatial-jp
resolution target: geospatial-jp
URI: https://www.geospatial.jp/ckan/dataset/6b8185aa-a89f-4887-810a-80c3d291a1c7/resource/78b3a6b9-f659-427a-bcc7-381074a38818/download/river-sabou.geojson
format / media type: geojson / None
AccessPlan: FileAccessPlan(kind='file', uri='https://www.geospatial.jp/ckan/dataset/6b8185aa-a89f-4887-810a-80c3d291a1c7/resource/78b3a6b9-f659-427a-bcc7-381074a38818/download/river-sabou.geojson', options=mappingproxy({}), archive=None)
metadata: {'title': '静岡県内の国管理施設-河川(砂防)', 'description': '未入力', 'publisher': '国土交通省', 'license': '政府標準利用規約'}
provenance: {'provider': 'geospatial-jp', 'dataset_identifier': '6b8185aa-a89f-4887-810a-80c3d291a1c7', 'resource_identifier': '78b3a6b9-f659-427a-bcc7-381074a38818', 'adapter': 'ckan', 'original_url': 'https://www.geospatial.jp/ckan/dataset/6b8185aa-a89f-4887-810a-80c3d291a1c7/resource/78b3a6b9-f659-427a-bcc7-381074a38818/download/river-sabou.geojson'}


## 利用者所有の Runtime で開き、地図として表示する

pyogrio は利用者が導入・登録する Execution Runtime です。Rhinestone は選択済みの URI と、Source が確定した属性だけを Runtime に渡します。開いた GeoDataFrame は Folium のベクターレイヤーにし、選択したデータの範囲へ地図を合わせます。

In [5]:
import folium
import pyogrio

app_with_pyogrio = configure(
    sources=(sources.GEOSPATIAL_JP,),
    dependencies={"pyogrio": pyogrio},
)
resource = app_with_pyogrio.resolve(selected)
frame = resource.open("pyogrio")

print("rows:", len(frame))
print("column count:", len(frame.columns))
if frame.crs is None:
    raise RuntimeError(
        "The selected Resource has no CRS; it cannot be placed on a web map."
    )

GSI_STANDARD_TILES = "https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png"
map_frame = frame.to_crs(epsg=4326)[["geometry"]]
west, south, east, north = map_frame.total_bounds
map_view = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=10,
    tiles=None,
    control_scale=True,
)
folium.TileLayer(
    tiles=GSI_STANDARD_TILES,
    attr="GSI Maps",
    name="GSI Standard Map",
    overlay=False,
    control=False,
).add_to(map_view)
folium.GeoJson(
    data=map_frame.__geo_interface__,
    name="Selected vector data",
    marker=folium.CircleMarker(
        radius=5, color="#2563eb", fill=True, fill_color="#2563eb", fill_opacity=0.8
    ),
).add_to(map_view)
if west != east and south != north:
    map_view.fit_bounds([[south, west], [north, east]])

map_view

rows: 178
column count: 29


## 境界

この Notebook は検索、解決、AccessPlan、Runtime への委譲を示します。操作できる地図と地理院タイルの背景タイルは pyogrio / GeoPandas / Folium の機能であり、Rhinestone は GIS 解析、形式変換、背景地図、archive の展開を実装しません。